# 04 · AVA-VLA 논문 최고 보고 설정 · 100K / batch 256
[논문 v4 Appendix D.1/Table 7](https://arxiv.org/html/2511.18960v4)은 사전학습 OpenVLA에서
100,000 gradient step·유효 batch 256으로 학습한 AVA-VLA의 LIBERO 평균 성공률 98.3%를 보고합니다.
이 숫자는 논문 보고값이며 우리 측정값이나 LIBERO+ 보장 성능이 아닙니다.
나머지 하이퍼파라미터는 Appendix A의 rank 32·K=4·2/3 경계 detach·LR 5e-4·10% warm-up·cosine을 따릅니다.
D.1이 나머지 조건을 모두 재명시하지 않았다는 점은 설정에 기록합니다.
우리 실행은 01의 LIBERO+ train 분할/정규화·query 간격 8을 쓰므로 데이터 조건이 다릅니다.
02의 zero-context FFT와 별도 실험입니다. 기존 AVA/OFT 가중치를 초기값으로 대신 쓰지 않습니다.
**준비 전용 기본값:** 설치·다운로드·파일럿·본 학습 플래그는 모두 False입니다. 모두 실행해도 대규모 작업은 시작하지 않습니다.
100K 목표를 8시간 예산에 맞춰 축소하지 않습니다. 8시간은 실행 세션별 상한이며 전체 실험에는 여러 세션이 필요할 수 있습니다.

In [ ]:
import time, os
PAPER_SESSION_START = globals().get('PAPER_SESSION_START',time.time())
os.environ['VLADE_DEADLINE_UNIX'] = str(PAPER_SESSION_START+8*3600)

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/colab/VLADE')  # 세 노트북에서 동일한 저장 폴더를 지정하십시오.
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = ROOT  # src/scripts/configs/experiments/third_party가 바로 들어 있는 실제 프로젝트 루트
EXPECTED_SOURCE_MANIFEST_SHA256 = '3caab51102f994611b4737aef199e9376bda7cade2bfa3e80ba1fde366509ef8'
def stage_source(project_root: Path, scratch_root: Path, expected_manifest_sha256: str) -> Path:
    """Verify Drive source files and stage them without copying data or weights."""
    project_root = project_root.resolve()
    manifest_relative = Path("experiments/manifests/colab_source.json")
    manifest_path = project_root / manifest_relative
    if not manifest_path.is_file():
        raise FileNotFoundError("Drive의 VLADE 폴더에 src/scripts/configs/experiments/third_party 일반 파일 구조를 먼저 복사하십시오: " + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_sha256:
        raise ValueError("노트북과 colab_source.json 버전이 다릅니다. 같은 버전의 일반 프로젝트 파일을 사용하십시오.")
    manifest = json.loads(manifest_bytes)
    code = scratch_root / ("vlade_colab_" + expected_manifest_sha256[:12])
    code = code.resolve()
    if code.is_relative_to(project_root) or project_root.is_relative_to(code):
        raise ValueError("실행 소스 임시 폴더와 Drive 프로젝트 폴더는 서로 겹칠 수 없습니다.")
    for relative, expected in manifest["files_sha256"].items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
            raise ValueError("잘못된 소스 상대 경로: " + relative)
        source = project_root / path
        target = code / path
        if not source.resolve().is_relative_to(project_root) or not target.resolve().is_relative_to(code):
            raise ValueError("프로젝트 밖의 소스 경로: " + relative)
        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise ValueError("Drive 소스가 없거나 변경되었습니다: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            shutil.copyfile(source, target)
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("복사된 소스 무결성 불일치: " + relative)
    # Publish the manifest only after every listed source file has been verified.
    staged_manifest = code / manifest_relative
    staged_manifest.parent.mkdir(parents=True, exist_ok=True)
    temporary_manifest = staged_manifest.with_suffix(".json.tmp")
    temporary_manifest.write_bytes(manifest_bytes)
    temporary_manifest.replace(staged_manifest)
    return code

CODE = stage_source(PROJECT_ROOT, Path('/content'), EXPECTED_SOURCE_MANIFEST_SHA256)
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('Drive:', ROOT, '\n실행 소스:', CODE)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'), HF_TOKEN_PATH='/content/vlade_hf_token')
def run(args, env=None):
    import signal, threading, time
    deadline = float((env or ENV).get('VLADE_DEADLINE_UNIX', '0'))
    if deadline and deadline - time.time() <= 60:
        raise RuntimeError('8시간 예산이 끝났습니다. 새 작업을 시작하지 않습니다.')
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timers = []
    if deadline:
        for delay, callback in [(max(0, deadline-time.time()-60), lambda: process.send_signal(signal.SIGINT)),
                                (max(0, deadline-time.time()), process.kill)]:
            timer = threading.Timer(delay, callback)
            timer.daemon = True
            timer.start()
            timers.append(timer)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('실행에 실패했거나 시간 예산이 끝났습니다. 위 결과를 확인하십시오.')
    finally:
        for timer in timers:
            timer.cancel()
UV = shutil.which('uv')
if UV is None:
    tool_env = Path('/content/vlade_uv_tool')
    tool_python = tool_env / 'bin/python'
    if not tool_python.is_file():
        run([sys.executable, '-m', 'venv', tool_env])
    run([tool_python, '-m', 'pip', 'install', 'uv'])
    UV = str(tool_env / 'bin/uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

# HF inputs are cached on the runtime SSD; Drive retains code and experiment records.
ASSET_CACHE = Path('/content/vlade_hf_assets')
ASSET_WORK_ROOT = Path('/content/vlade_asset_work')
RUNS_ROOT = Path('/content/vlade_runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
ENV.update(VLADE_ASSET_CACHE=str(ASSET_CACHE),
           VLADE_ASSET_REGISTRY=str(CODE/'configs/storage/hf_assets.lock.json'),
           HF_HOME='/content/vlade_hf_home', HF_TOKEN_PATH='/content/vlade_hf_token')
os.environ.update({key: ENV[key] for key in ('VLADE_ASSET_CACHE','VLADE_ASSET_REGISTRY')})
_HF_REGISTRY_READY = False

def hf_input(key, python=None, *, fetch=True):
    global _HF_REGISTRY_READY
    from vlade.storage.assets import asset_record, cache_path
    if fetch and not _HF_REGISTRY_READY:
        if python is None:
            raise ValueError('Select the prepared Python environment before fetching HF assets')
        registry = ASSET_CACHE / 'registry.json'
        run([python, 'scripts/data/hf_assets.py', 'refresh-registry', '--output', registry], env=ENV)
        ENV['VLADE_ASSET_REGISTRY'] = str(registry)
        os.environ['VLADE_ASSET_REGISTRY'] = str(registry)
        _HF_REGISTRY_READY = True
    record = asset_record(key, Path(ENV['VLADE_ASSET_REGISTRY']))
    target = cache_path(key, record, ASSET_CACHE)
    if fetch:
        if python is None:
            raise ValueError('Select the prepared CPU/model Python environment before fetching HF assets')
        run([python, 'scripts/data/hf_assets.py', 'fetch', key,
             '--registry', ENV['VLADE_ASSET_REGISTRY'], '--cache', ASSET_CACHE], env=ENV)
    return target

def hf_path(value, python=None, *, fetch=True):
    value = str(value)
    return hf_input(value[len('hfasset:'):].strip('/\\'), python, fetch=fetch) if value.startswith('hfasset:') else Path(value)

In [ ]:
# HF 설정 후 기존 학습 셀을 실행하면 checkpoint 저장 시 자동 전송합니다.
# 저장소는 HF 웹에서 Private로 먼저 만들고 해당 저장소 쓰기 토큰을 Secrets에 등록하십시오.
if 'ENV' not in globals():
    ENV = os.environ.copy()
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
ENV['HF_TOKEN'] = os.environ['HF_TOKEN']
import datetime, uuid
ENABLE_HUB_UPLOAD = globals().get('ENABLE_HUB_UPLOAD', True)
HF_REPO_ID = 'VLADE-konkuk/VLADE-checkpoints'  # 조직 저장소; 이전 런타임 값도 갱신합니다.
HF_RUN_GROUP = globals().get('HF_RUN_GROUP', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])  # 재개 시 기존 group 지정
if ENABLE_HUB_UPLOAD:
    if not HF_RUN_GROUP:
        raise ValueError('HF_RUN_GROUP을 지정하십시오. pilot/main은 출력 폴더명으로 구분합니다.')
    os.environ['VLADE_HUB_REPO'] = HF_REPO_ID
    os.environ['VLADE_HUB_RUN_ID'] = HF_RUN_GROUP
    os.environ['HF_TOKEN_PATH'] = '/content/vlade_hf_token'
    # 모델 캐시는 기존 위치를 유지하고 토큰 파일은 런타임 로컬에만 둡니다.
    ENV.update(HF_TOKEN=os.environ['HF_TOKEN'], VLADE_HUB_REPO=HF_REPO_ID,
               VLADE_HUB_RUN_ID=HF_RUN_GROUP, HF_TOKEN_PATH='/content/vlade_hf_token')
    print('Private Hub 업로드 설정:', HF_REPO_ID, HF_RUN_GROUP)
else:
    for key in ('VLADE_HUB_REPO', 'VLADE_HUB_RUN_ID'):
        ENV.pop(key, None)
        os.environ.pop(key, None)
    print('Hub 업로드 꺼짐: HF_RUN_GROUP 설정 후 ENABLE_HUB_UPLOAD=True로 이 셀을 다시 실행하십시오.')

In [ ]:
RUN_INSTALL = False
RUN_DOWNLOAD = False
RUN_PILOT = False
RUN_PAPER_TRAINING = False
config = json.loads((CODE/'configs/experiments/ava_paper_best.json').read_text())
config['initial_checkpoint'] = config['initial_checkpoint']
config['data'] = config['data']
config['output'] = str(RUNS_ROOT/config['output'].removeprefix('runs/'))
CONFIG_PATH = ROOT/'run-configs/ava-paper-best-seed42.json'
CONFIG_PATH.parent.mkdir(exist_ok=True)
if CONFIG_PATH.exists() and json.loads(CONFIG_PATH.read_text()) != config:
    raise ValueError('기존 04 실행 설정과 다릅니다. 기존 설정/결과를 덮어쓰지 않습니다.')
CONFIG_PATH.write_text(json.dumps(config,indent=2))
print(json.dumps(config,indent=2,ensure_ascii=False))
print('1 optimizer step = 256 K=4 sequences = 1024 query 예측')

## 환경 설치 (선택)
02와 같은 검증된 AVA Transformers 패치를 사용합니다. K=4의 두 프레임 BPTT를 위해
non-reentrant gradient checkpointing을 사용합니다. 실제 GPU 메모리 적합성은 파일럿에서 확인해야 합니다.
준비만 할 때는 RUN_INSTALL=False를 유지하십시오.

In [ ]:
if RUN_INSTALL:
    import threading
    from google.colab import runtime
    _paper_release_timer = threading.Timer(max(0,PAPER_SESSION_START+8*3600-time.time()),runtime.unassign)
    _paper_release_timer.daemon = True
    _paper_release_timer.start()
    AVA_ENV_READY = False
    AVA_PY = None
    import tempfile, tomllib, zipfile
    AVA_VENV = Path('/content') / 'vlade_ava_fft_py311'
    AVA_PATCH_REL = 'configs/models/teacher/transformers-avavla-a03eee5-checkpoint-extra-attn.patch'
    AVA_PATCH_SHA = '9f0e46c9e93d62a92af950aeee712191cfcf314c1e636a9608fddcb57ff6087d'
    AVA_TRANSFORMERS_REV = 'a03eee5da1794c45f5a911b60a9a4545e6e0ed1f'
    AVA_BASE_SHA = '62f22856fe7b8459d6d61d4676b0075b65724979f84859ac475dbe36c10a3d80'
    AVA_INSTALLED_SHA = '0a9c25300f3358f87af1719199608af3c27567dba0425690327fccd44cfb516b'
    AVA_VERIFY_SOURCE = "import ast, hashlib, importlib, json, sys\nfrom pathlib import Path\nexpected_sha, expected_prefix = sys.argv[1:3]\nprefix = Path(sys.prefix).resolve()\nif sys.prefix == sys.base_prefix or prefix != Path(expected_prefix).resolve():\n    raise RuntimeError('AVA verification requires the dedicated virtual environment')\nmodule = importlib.import_module('transformers.models.llama.modeling_llama')\nsource = Path(module.__file__).resolve()\nif not source.is_relative_to(prefix) or source.name != 'modeling_llama.py':\n    raise RuntimeError('Transformers was imported outside the AVA environment')\npayload = source.read_bytes()\nif hashlib.sha256(payload).hexdigest() != expected_sha:\n    raise RuntimeError('Installed AVA Transformers source SHA mismatch')\ntree = ast.parse(payload)\nmodel = next(node for node in tree.body if isinstance(node, ast.ClassDef) and node.name == 'LlamaModel')\nforward = next(node for node in model.body if isinstance(node, ast.FunctionDef) and node.name == 'forward')\ncalls = [node for node in ast.walk(forward) if isinstance(node, ast.Call)\n         and isinstance(node.func, ast.Attribute) and node.func.attr == '_gradient_checkpointing_func']\nexpected = 'extra_attn_weights if extra_layer_ids and layer_idx in extra_layer_ids else None'\nif len(calls) != 1 or len(calls[0].args) != 9 or ast.unparse(calls[0].args[-1]) != expected:\n    raise RuntimeError('Installed checkpoint attention argument was not preserved')\nprint(json.dumps({'source': str(source), 'sha256': expected_sha, 'venv': str(prefix)}))"
    _VLADE_AVA_BASE_RUN = getattr(run, '_vlade_base_run', run)

    def verify_ava_transformers(python):
        _VLADE_AVA_BASE_RUN([python, '-c', AVA_VERIFY_SOURCE, AVA_INSTALLED_SHA, AVA_VENV])

    def require_ava_environment():
        if not AVA_ENV_READY or AVA_PY is None:
            raise RuntimeError('AVA patched environment is not verified; rerun the installation cell')
        verify_ava_transformers(AVA_PY)

    def run(args, env=None):
        command = [str(value) for value in args]
        if command and (command[0] == str(AVA_VENV / 'bin/python')
                        or any(Path(value).name == 'run_fft.py' for value in command)):
            require_ava_environment()
        return _VLADE_AVA_BASE_RUN(args, env=env)

    run._vlade_base_run = _VLADE_AVA_BASE_RUN
    patch = CODE / AVA_PATCH_REL
    source_manifest = json.loads((CODE / 'experiments/manifests/colab_source.json').read_text())
    if source_manifest['files_sha256'].get(AVA_PATCH_REL) != AVA_PATCH_SHA:
        raise RuntimeError('Staged source manifest does not pin the approved AVA patch')
    if not patch.is_file() or hashlib.sha256(patch.read_bytes()).hexdigest() != AVA_PATCH_SHA:
        raise RuntimeError('Staged AVA patch is missing or has the wrong SHA')

    build_root = Path(tempfile.mkdtemp(prefix='vlade-ava-transformers-', dir=AVA_VENV.parent))
    source = build_root / 'source'
    source.mkdir()
    build_env = ENV.copy()
    build_env.update(UV_CACHE_DIR=str(build_root / 'uv-cache'), TMPDIR=str(build_root / 'tmp'))
    (build_root / 'tmp').mkdir()
    fork = 'https://github.com/LiAuto-DSR/transformers-avavla.git'
    _VLADE_AVA_BASE_RUN(['git', '-C', source, 'init'], env=build_env)
    _VLADE_AVA_BASE_RUN(['git', '-C', source, 'fetch', '--depth', '1', fork, AVA_TRANSFORMERS_REV], env=build_env)
    _VLADE_AVA_BASE_RUN(['git', '-C', source, 'checkout', '--detach', 'FETCH_HEAD'], env=build_env)
    revision = subprocess.check_output(['git', '-C', str(source), 'rev-parse', 'HEAD'], text=True).strip()
    if revision != AVA_TRANSFORMERS_REV:
        raise RuntimeError('AVA Transformers checkout revision mismatch')
    llama = source / 'src/transformers/models/llama/modeling_llama.py'
    if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_BASE_SHA:
        raise RuntimeError('Unpatched AVA Transformers source SHA mismatch')
    _VLADE_AVA_BASE_RUN(['git', '-C', source, 'apply', '--check', patch], env=build_env)
    _VLADE_AVA_BASE_RUN(['git', '-C', source, 'apply', patch], env=build_env)
    if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_INSTALLED_SHA:
        raise RuntimeError('Patched AVA Transformers source SHA mismatch')
    wheels = build_root / 'wheels'
    wheels.mkdir()
    _VLADE_AVA_BASE_RUN([UV, 'build', '--wheel', '--python', '3.11', '--out-dir', wheels, source], env=build_env)
    wheel_files = list(wheels.glob('transformers-*.whl'))
    if len(wheel_files) != 1:
        raise RuntimeError('Expected exactly one patched Transformers wheel')
    wheel = wheel_files[0]
    with zipfile.ZipFile(wheel) as archive:
        wheel_source = archive.read('transformers/models/llama/modeling_llama.py')
    if hashlib.sha256(wheel_source).hexdigest() != AVA_INSTALLED_SHA:
        raise RuntimeError('Built Transformers wheel does not contain the approved patch')

    upstream = CODE / 'third_party/ava_vla'
    dependencies = tomllib.loads((upstream / 'pyproject.toml').read_text())['project']['dependencies']
    dependencies = [dep for dep in dependencies if not dep.startswith(('transformers @', 'dlimp @', 'huggingface_hub', 'accelerate'))]
    dependencies += [
        str(wheel),
        'dlimp @ git+https://github.com/moojink/dlimp_openvla.git@040105d256bd28866cc6620621a3d5f7b6b91b46',
        'numpy==1.26.4', 'tensorflow-metadata==1.15.0', 'bitsandbytes==0.43.3',
        'huggingface_hub==0.36.2', 'accelerate==1.15.0']
    candidate_python = environment('vlade_ava_fft_py311', '3.11', dependencies)
    if str(candidate_python) != str(AVA_VENV / 'bin/python'):
        raise RuntimeError('Refusing to install outside the dedicated AVA environment')
    _VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python,
                        '--no-deps', '--force-reinstall', wheel], env=build_env)
    _VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python, '--no-deps', '-e', upstream])
    verify_ava_transformers(candidate_python)
    _VLADE_AVA_BASE_RUN([candidate_python, '-c', "import torch,transformers,tensorflow_datasets; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,transformers.__version__,torch.cuda.get_device_name(0),torch.cuda.get_device_properties(0).total_memory/2**30)"])
    freeze = subprocess.check_output([str(UV), 'pip', 'freeze', '--python', str(candidate_python)], text=True)
    (ROOT / 'ava_environment_freeze.txt').write_text(freeze)
    (AVA_VENV / 'vlade_transformers_patch.json').write_text(json.dumps({
        'base_commit': AVA_TRANSFORMERS_REV, 'patch_sha256': AVA_PATCH_SHA,
        'installed_source_sha256': AVA_INSTALLED_SHA,
        'wheel_sha256': hashlib.sha256(wheel.read_bytes()).hexdigest()}, indent=2))
    AVA_PY = candidate_python
    AVA_ENV_READY = True
    print('AVA managed Transformers patch verified; FFT environment ready')
else:
    print('환경 설치 미실행: RUN_INSTALL=False')

In [ ]:
if RUN_INSTALL:
    import threading
    from google.colab import runtime
    if globals().get('_paper_release_timer'):
        _paper_release_timer.cancel()
    _paper_release_timer = threading.Timer(max(0,PAPER_SESSION_START+8*3600-time.time()),runtime.unassign)
    _paper_release_timer.daemon = True
    _paper_release_timer.start()
    print('8시간 세션 마감 시 런타임 반환을 요청합니다.')

In [ ]:
if RUN_DOWNLOAD:
    require_ava_environment()
    program = "from pathlib import Path; import sys; sys.path.insert(0,'src'); from vlade.data.colab_assets import download; download(sys.argv[1],sys.argv[2],Path(sys.argv[3]))"
    hf_input('openvla_pretrained', AVA_PY)
    hf_input('prepared_d_seed42', AVA_PY)
else:
    print('가중치 다운로드 미실행: RUN_DOWNLOAD=False (사전학습 OpenVLA 약 15.1 GB)')

In [ ]:
if RUN_PILOT:
    require_ava_environment()
    run([AVA_PY,'scripts/train/run_paper_ava.py','--config',CONFIG_PATH,'--preflight'])
    pilot = Path(config['output']+'-pilot')
    if not (pilot/'pilot_report.json').is_file():
        run([AVA_PY,'scripts/train/run_paper_ava.py','--config',CONFIG_PATH,'--pilot'])
    print((pilot/'pilot_report.json').read_text())
else:
    print('K=4 backward·LoRA·정규화·저장/재로딩 파일럿 미실행')

## 본 학습/재개 (선택)
파일럿은 1 sequence로 K=4 역전파·저장/재로딩 경로를 점검하며 batch 256의 성능 검증은 아닙니다.
본 학습은 microbatch 1×누적 256을 사용합니다. 첫 step·5,000 step 간격·세션 종료 전에
LoRA adapter, head/proprio, AVA generator, optimizer/scheduler/RNG와 sequence cursor를 Drive에 저장합니다.
100,000 step과 warm-up 10,000은 재개 중 바꾸지 않습니다. 중단 시 데이터 cursor를 복원합니다.
CPU 테스트만 통과한 자체 학습 어댑터이며 저자 학습 코드 재현 또는 98.3% 달성을 검증한 것은 아닙니다.
기존 02의 T1/cache 도구는 FFT 형식 전용이므로 04 adapter checkpoint를 직접 입력하지 마십시오.

In [ ]:
if RUN_PAPER_TRAINING:
    require_ava_environment()
    # 재개할 때는 이 경로만 지정하고 목표 step/LR schedule은 유지합니다.
    RESUME_CHECKPOINT = ''
    command = [AVA_PY,'scripts/train/run_paper_ava.py','--config',CONFIG_PATH]
    if RESUME_CHECKPOINT:
        command += ['--resume',RESUME_CHECKPOINT]
    else:
        report = Path(config['output']+'-pilot')/'pilot_report.json'
        if not report.is_file():
            raise RuntimeError('K=4 실제 파일럿을 먼저 실행하십시오.')
        check = "import json,sys; from pathlib import Path; sys.path.insert(0,'src'); from vlade.training.paper_ava import require_pilot; require_pilot(json.loads(Path(sys.argv[1]).read_text()))"
        run([AVA_PY,'-c',check,CONFIG_PATH,report])
    run(command)
else:
    print('준비 완료. 본 학습은 실행하지 않았습니다. RUN_PAPER_TRAINING=False')